In [1]:
import json
import torch
import torch.nn as nn

class PhotoAssistantNet(nn.Module):
    def __init__(self, input_size=7, num_classes=4):
        super().__init__()
        self.fc1 = nn.Linear(input_size, 32)
        self.fc2 = nn.Linear(32, 16)
        self.fc3 = nn.Linear(16, num_classes)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)
        return x

model = PhotoAssistantNet()
model.load_state_dict(torch.load("photo_assistant_model.pth", map_location="cpu"))
model.eval()

with open("photo_assistant_metadata.json", "r") as f:
    meta = json.load(f)

state = model.state_dict()

def array_to_c(name, arr):
    flat = arr.flatten().tolist()
    values = ", ".join(f"{x:.8f}f" for x in flat)
    return f"const float {name}[] = {{{values}}};\n"

def scalar_array_to_c(name, arr):
    values = ", ".join(f"{x:.8f}f" for x in arr)
    return f"const float {name}[] = {{{values}}};\n"

header = []
header.append("#ifndef MODEL_WEIGHTS_H")
header.append("#define MODEL_WEIGHTS_H")
header.append("")

header.append("const int INPUT_SIZE = 7;")
header.append("const int H1_SIZE = 32;")
header.append("const int H2_SIZE = 16;")
header.append("const int OUTPUT_SIZE = 4;")
header.append("")

header.append(array_to_c("FC1_WEIGHT", state["fc1.weight"].numpy()))
header.append(array_to_c("FC1_BIAS", state["fc1.bias"].numpy()))
header.append(array_to_c("FC2_WEIGHT", state["fc2.weight"].numpy()))
header.append(array_to_c("FC2_BIAS", state["fc2.bias"].numpy()))
header.append(array_to_c("FC3_WEIGHT", state["fc3.weight"].numpy()))
header.append(array_to_c("FC3_BIAS", state["fc3.bias"].numpy()))

header.append(scalar_array_to_c("SCALER_MEAN", meta["scaler_mean"]))
header.append(scalar_array_to_c("SCALER_SCALE", meta["scaler_scale"]))

labels = meta["labels"]
label_lines = ", ".join([f"\"{x}\"" for x in labels])
header.append(f"const char* CLASS_NAMES[] = {{{label_lines}}};")

header.append("")
header.append("#endif")

with open("model_weights.h", "w") as f:
    f.write("\n".join(header))

print("Created model_weights.h")

Created model_weights.h
